In [0]:
%run ./_bootstrap

In [0]:
import os
import json
import tempfile
import hashlib
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import mlflow
import mlflow.pyfunc
from mlflow.tracking import MlflowClient
import joblib
from helpers.mlflow_retriever import load_retriever_artifacts

In [0]:
spark = SparkSession.builder.getOrCreate()
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA med")

In [0]:
client = MlflowClient()
exp_name = "/Users/dhrutigandhi.05@gmail.com/medibot_retrieval"
exp = client.get_experiment_by_name(exp_name) # get experiment if it already exists

# create experiment if it does not exist
if exp is None:
    exp_id = client.create_experiment(exp_name)
else:
    exp_id = exp.experiment_id

# set experiment by id
mlflow.set_experiment(experiment_id=exp_id)
print("Using experiment:", exp_name)
print("Experiment id:", exp_id)


In [0]:
# get the latest train run id
latest_row = (
  spark.table("workspace.med.retriever_registry")
       .orderBy(F.col("created_at").desc())
       .limit(1)
       .collect()
)

if not latest_row:
    raise RuntimeError("No rows in workspace.med.retriever_registry. Train job has not recorded a run id yet.")

TRAIN_RUN_ID = latest_row[0]["train_run_id"]
print("Using latest TRAIN_RUN_ID:", TRAIN_RUN_ID)

vectorizer, knn, chunk_ids, train_metadata = load_retriever_artifacts(run_id=TRAIN_RUN_ID, artifact_path="retriever")

print("num chunk_ids:", len(chunk_ids))
print("train_metadata keys:", list(train_metadata.keys()))

In [0]:
doc_chunks_df = spark.table("workspace.med.doc_chunks") # load the chunk table from delta
serve_cols = ["chunk_id", "doc_id", "source", "category", "title", "chunk_text"] # choose the columns we want to return from the endpoint
filtered_df = doc_chunks_df.select(*serve_cols).where(F.col("chunk_id").isin(chunk_ids)) # filter to only chunk_ids in the training run
chunks_pdf = filtered_df.toPandas() # convert to pandas for serving
pos_map = {cid: i for i, cid in enumerate(chunk_ids)} # build a position map to restore the exact training order
chunks_pdf = chunks_pdf[chunks_pdf["chunk_id"].isin(pos_map)] # keep only rows that exist in pos_map

# add position and sort
chunks_pdf["__pos"] = chunks_pdf["chunk_id"].map(pos_map)
chunks_pdf = chunks_pdf.sort_values("__pos").drop(columns=["__pos"]).reset_index(drop=True)

# sanity checks to confirm alignment
print("chunks_pdf rows:", len(chunks_pdf))
print("first matches:", chunks_pdf.iloc[0]["chunk_id"] == chunk_ids[0])
print("last matches:", chunks_pdf.iloc[-1]["chunk_id"] == chunk_ids[-1])

In [0]:
# build a dictionary to include with the serving model
serve_metadata = {
    "catalog": "workspace",
    "schema": "med",
    "source_table": "workspace.med.doc_chunks",
    "num_chunks": int(chunks_pdf.shape[0]),
    "train_run_id": str(TRAIN_RUN_ID),
}

tmpdir = tempfile.mkdtemp()

# define output paths
vec_path = os.path.join(tmpdir, "vectorizer.joblib")
knn_path = os.path.join(tmpdir, "knn.joblib")
chunks_path = os.path.join(tmpdir, "chunks.parquet")
meta_path = os.path.join(tmpdir, "metadata.json")

# save vectorizer and knn to disk
joblib.dump(vectorizer, vec_path)
joblib.dump(knn, knn_path)

chunks_pdf.attrs.pop("metrics", None)

# save chunk data to parquet file format
chunks_pdf.to_parquet(chunks_path, index=False)
print("wrote parquet:", chunks_path)

# save metadata json
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(serve_metadata, f)

print("tmpdir:", tmpdir)
print("saved chunks rows:", len(chunks_pdf))

In [0]:
# define a pyfunc model that loads artifacts and runs retrieval
class RetrieverPyfunc(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.vectorizer = joblib.load(context.artifacts["vectorizer"]) # load vectorizer from artifacts
        self.knn = joblib.load(context.artifacts["knn"]) # load knn index from artifacts
        self.chunks = pd.read_parquet(context.artifacts["chunks"]) # load chunks dataframe from artifacts

        # load metadata json from artifacts
        with open(context.artifacts["metadata"], "r", encoding="utf-8") as f:
            self.metadata = json.load(f)

    def predict(self, context, model_input):
        # handle dict input
        if isinstance(model_input, dict):
            question = model_input.get("question", "")
            top_k = int(model_input.get("top_k", 5))
            pool_k = int(model_input.get("pool_k", 50))
            max_dist = float(model_input.get("max_dist", 0.85))
        else:
            # handle dataframe input
            question = str(model_input.iloc[0].get("question", ""))
            top_k = int(model_input.iloc[0].get("top_k", 5))
            pool_k = int(model_input.iloc[0].get("pool_k", 50))
            max_dist = float(model_input.iloc[0].get("max_dist", 0.85))

        question = (question or "").strip() # normalize question

        # return empty output if no question
        if not question:
            return pd.DataFrame([{"chunks": [], "metadata": self.metadata}])

        pool_k = min(pool_k, len(self.chunks)) # bound pool size
        q_vec = self.vectorizer.transform([question]) # vectorize the question
        distances, indices = self.knn.kneighbors(q_vec, n_neighbors=pool_k) # retrieve nearest neighbors

        # dedupe sets
        seen_doc = set()
        seen_text = set()
        out = []

        # loop candidates in ranked order
        for idx, dist in zip(indices[0], distances[0]):
            # skip weak matches
            if dist > max_dist:
                continue

            row = self.chunks.iloc[int(idx)] # fetch row by index

            # read values
            doc_id = row["doc_id"]
            txt = row["chunk_text"] or ""

            # dedupe identical chunk text
            h = hashlib.md5(txt.encode("utf-8")).hexdigest()
            if h in seen_text:
                continue

            # dedupe within same document
            if doc_id in seen_doc:
                continue

            # mark seen
            seen_text.add(h)
            seen_doc.add(doc_id)

            # append output record
            out.append({
                "chunk_id": row["chunk_id"],
                "doc_id": row["doc_id"],
                "source": row["source"],
                "category": row["category"],
                "title": row["title"],
                "cosine_distance": float(dist),
                "cosine_similarity": float(1.0 - float(dist)),
                "chunk_text": row["chunk_text"],
            })

            # stop when we have enough
            if len(out) >= top_k:
                break

        # return a one row dataframe for serving compatibility
        return pd.DataFrame([{"chunks": out, "metadata": self.metadata}])


In [0]:
# log a deployable MLflow pyfunc model
with mlflow.start_run(run_name="retriever_pyfunc_bundle") as run:
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=RetrieverPyfunc(),
        artifacts={
            "vectorizer": vec_path,
            "knn": knn_path,
            "chunks": chunks_path,
            "metadata": meta_path,
        },
        pip_requirements=[
            "mlflow",
            "joblib",
            "pandas",
            "numpy",
            "scikit-learn",
            "pyarrow",
        ],
    )

    # print model uri
    model_uri = f"runs:/{run.info.run_id}/model"
    print("MODEL_URI:", model_uri)
    print("RUN_ID:", run.info.run_id)

In [0]:
# load the model back for local testing
loaded = mlflow.pyfunc.load_model(model_uri)

# create test input
test_in = pd.DataFrame([{
    "question": "what is ibuprofen used for",
    "top_k": 5,
    "pool_k": 50,
    "max_dist": 0.85
}])

# run prediction
test_out = loaded.predict(test_in)

# display raw output
display(test_out)

In [0]:
# unpack chunks from the one row dataframe
chunks = test_out.iloc[0]["chunks"]

# print each chunk
for i, c in enumerate(chunks, start=1):
    print(i, c["source"], c["title"], "dist=", c["cosine_distance"], "chunk_id=", c["chunk_id"])
    print((c["chunk_text"] or "")[:300])
    print("-" * 100)